# 11 – Ratings

Esplorazione e data cleaning del dataset `ratings.csv`.

> **Nota:** il file contiene ~124 milioni di righe. Il caricamento e le operazioni richiedono tempo e memoria significativi.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL (FK → `profiles.csv`) |
| `anime_id` | ID dell'anime su MAL (FK → `details.csv`) |
| `status` | Stato di visione (`watching`, `completed`, `on_hold`, `dropped`, `plan_to_watch`) |
| `score` | Voto dell'utente (0–10; 0 = non votato) |
| `is_rewatching` | Indica se l'utente sta rivedendo l'anime (0/1/NaN) |
| `num_watched_episodes` | Numero di episodi guardati |

## 1. Import e caricamento dati

In [30]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_rt = pd.read_csv('../datasets/ratings.csv')
print(f'Shape: {df_rt.shape}')
df_rt.info()
df_rt.head()

Shape: (124298357, 6)
<class 'pandas.DataFrame'>
RangeIndex: 124298357 entries, 0 to 124298356
Data columns (total 6 columns):
 #   Column                Dtype  
---  ------                -----  
 0   username              str    
 1   anime_id              int64  
 2   status                str    
 3   score                 int64  
 4   is_rewatching         float64
 5   num_watched_episodes  int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 5.6 GB


,username,anime_id,status,score,is_rewatching,num_watched_episodes
0,--------788,30276,watching,7,0.0,3
1,--------788,28851,completed,7,0.0,1
2,--------788,41168,completed,7,0.0,1
3,--------788,22199,completed,10,0.0,24
4,--------788,16498,completed,10,0.0,25


**Osservazioni iniziali:**
- Il dataset contiene ~124 milioni di righe e 6 colonne.
- `username` presenta 7 righe nulle (rimosse in §1.2).
- `status` contiene 16.690 righe con valore `'unknown'` (rimosse in §1.3).
- `is_rewatching` ha dtype `float64` con valori `0.0`, `1.0` e `NaN` → conversione a `boolean` nullable in §2.5.
- I tipi delle restanti colonne sono adeguati.

## 1.1 Rimozione duplicati esatti

La dimensione del dataset (~124M righe) rende la verifica completa delle righe duplicate proibitiva in termini di memoria. Verifichiamo su un campione rappresentativo di 2 milioni di righe; la chiave composita `(username, anime_id)` è verificata sull'intero dataset in §2.7.

In [31]:
n_originale = len(df_rt)

df_dup_sample = df_rt.sample(n=2_000_000, random_state=42)
n_dup_sample = df_dup_sample.duplicated(keep=False).sum()
print(f'Duplicati esatti nel campione di 2.000.000 righe: {n_dup_sample:,}')
if n_dup_sample == 0:
    print('→ Nessun duplicato esatto nel campione, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati: verificare sull\'intero dataset.')

Duplicati esatti nel campione di 2.000.000 righe: 0
→ Nessun duplicato esatto nel campione, nessuna operazione richiesta.


Nessun duplicato esatto rilevato nel campione. Il dataset è considerato privo di duplicati esatti.

## 1.2 Rimozione righe senza `username`

Sono presenti 7 righe consecutive con `username` null (righe 100.051.758–100.051.764). Senza identificatore utente la riga non è collegabile a nessun profilo → rimozione.

In [32]:
print(f'Righe con username null: {df_rt["username"].isna().sum()}')
print()
print('Righe da rimuovere:')
print(df_rt[df_rt['username'].isna()])

df_rt.dropna(subset=['username'], inplace=True)
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

Righe con username null: 7

Righe da rimuovere:
          username  anime_id         status  score  is_rewatching  \
100051758      NaN      1482       watching      9            0.0   
100051759      NaN      1735       watching      9            0.0   
100051760      NaN       121      completed     10            0.0   
100051761      NaN       136      completed      8            0.0   
100051762      NaN       269        on_hold      7            0.0   
100051763      NaN      1818        on_hold      7            0.0   
100051764      NaN      1535  plan_to_watch      6            0.0   

           num_watched_episodes  
100051758                    35  
100051759                    21  
100051760                    51  
100051761                    62  
100051762                    33  
100051763                    11  
100051764                     4  

Righe dopo rimozione: 124,298,350


Rimosse 7 righe senza `username`. Senza identificatore utente la riga non è collegabile a nessun profilo.

## 1.3 Rimozione righe con `status = 'unknown'`

16.690 righe presentano `status = 'unknown'`: hanno sempre `score = 0` e `num_watched_episodes = 0`, quindi non contengono informazioni utili → rimozione.

In [33]:
mask_unknown = df_rt['status'] == 'unknown'
print(f'Righe con status=unknown : {mask_unknown.sum():,}')
print()
print('Caratteristiche di queste righe:')
print(df_rt[mask_unknown][['score', 'num_watched_episodes']].describe())

df_rt = df_rt[~mask_unknown].copy()
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

Righe con status=unknown : 16,690

Caratteristiche di queste righe:
              score  num_watched_episodes
count  16690.000000          16690.000000
mean       0.306111              2.807969
std        1.561780             27.398930
min        0.000000              0.000000
25%        0.000000              0.000000
50%        0.000000              0.000000
75%        0.000000              1.000000
max       10.000000           1244.000000

Righe dopo rimozione: 124,281,660


Rimosse 16.690 righe con `status = 'unknown'`. Queste righe avevano sempre `score = 0` e `num_watched_episodes = 0` e non contenevano informazioni utili.

## 2. Analisi colonna per colonna

### 2.1 `username`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `username` di `profiles.csv`.

I valori duplicati sono **attesi**: lo stesso utente ha più righe (una per ogni anime in lista).

I controlli rilevanti sono:
- **Valori nulli**: già rimossi in §1.2.
- **Integrità referenziale**: ogni username deve esistere in `profiles_clean.csv`.

Usiamo `check_fk` al posto di `analyze`.

In [34]:
df_rt['username'] = df_rt['username'].str.strip()
df_profiles = pd.read_csv('../datasets_cleaned/profiles_clean.csv', usecols=['username'])

mask_orphan_user = check_fk(df_rt['username'], df_profiles['username'], child_df=df_rt)

print(f'Null in username               : {df_rt["username"].isna().sum()}')
print(f'Duplicati in username (attesi) : {df_rt["username"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        username
  Colonna PK  (tabella padre)         username
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       124,281,660
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            124,281,660  (100.00%)
  Valori unici nella PK padre         335,476

  ✓  Righe con FK valida              123,641,496  (99.48%)
  ✗  Righe orfane (FK non in PK)      640,164  (0.52%)
     → ID orfani unici                1,673

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

640,164 riga/e (0.52%) violano l'integrità referenziale.

Campione righe orfane (prime 10)
────────────────────────────────────────────────────────────────────────────────

     username  anime_id     status  score  is_rewatching  num_wat

**Osservazioni:**
- **Nessun valore nullo**: rimossi in §1.2.
- **Integrità referenziale**: verificare l'output sopra per eventuali righe orfane.

Se presenti, le righe orfane vengono rimosse nella cella seguente.

In [35]:
if mask_orphan_user.any():
    n_orfane = mask_orphan_user.sum()
    df_rt = df_rt[~mask_orphan_user].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane:,}')
    print(f'Righe rimanenti      : {len(df_rt):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

Righe orfane rimosse : 640,164
Righe rimanenti      : 123,641,496


### 2.2 `anime_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: lo stesso anime compare in più righe (uno per utente).

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID deve esistere in `details_clean.csv`.

Usiamo `check_fk` al posto di `analyze`.

In [36]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv', usecols=['mal_id'])

mask_orphan_anime = check_fk(df_rt['anime_id'], df_details['mal_id'], child_df=df_rt)

print(f'Null in anime_id               : {df_rt["anime_id"].isna().sum()}')
print(f'Duplicati in anime_id (attesi) : {df_rt["anime_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        anime_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       123,641,496
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            123,641,496  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              123,335,305  (99.75%)
  ✗  Righe orfane (FK non in PK)      306,191  (0.25%)
     → ID orfani unici                326

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

306,191 riga/e (0.25%) violano l'integrità referenziale.

Campione righe orfane (prime 10)
────────────────────────────────────────────────────────────────────────────────

            username  anime_id         status  score  is_rewatching  n

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime valido.
- **Integrità referenziale**: verificare l'output sopra per eventuali righe orfane.

Se presenti, le righe orfane vengono rimosse nella cella seguente.

In [37]:
if mask_orphan_anime.any():
    n_orfane = mask_orphan_anime.sum()
    df_rt = df_rt[~mask_orphan_anime].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane:,}')
    print(f'Righe rimanenti      : {len(df_rt):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

Righe orfane rimosse : 306,191
Righe rimanenti      : 123,335,305


### 2.3 `status`

In [38]:
df_rt['status'] = df_rt['status'].str.strip()
analyze(df_rt['status'])


  Nome serie                     status
  dtype                          str
  Dimensione                     123,335,305
  Range indice                   0 … 123335304
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,335,305
  Valori non nulli               123,335,305  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   5  (0.00%)
  Valori duplicati               123,335,300  (100.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  7  caratteri
  Lunghezza max                  13  caratteri
  Lunghezza media                9.85  caratteri
  Lunghezza mediana              9.0000  caratteri
  Dev. standard lunghezza        1.91
  IQR lunghezza                  4.00

**Nessuna pulizia necessaria.**  
Nessun null, 5 valori distinti dopo la rimozione di `'unknown'` (passo 1.2). La distribuzione riflette il comportamento tipico degli utenti MAL: `completed` è il più frequente.

### 2.4 `score`

In [39]:
analyze(df_rt['score'])


  Nome serie                     score
  dtype                          int64
  Dimensione                     123,335,305
  Range indice                   0 … 123335304
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,335,305
  Valori non nulli               123,335,305  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           54,235,027  (43.97%)
  Positivi                       69,100,278   (56.03%)
  Negativi                       0   (0.00%)
  Valori unici                   11  (0.00%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            10
  Range                          10
  Media                          4.0953
  Mediana                        5.0000
  Moda/e                         0
  Dev. standar

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`, valori nell'intervallo 0–10. Il valore `0` indica che l'utente **non ha assegnato un voto** (~44% delle righe): è un comportamento atteso su MAL (un anime può essere in lista senza essere valutato).

### 2.5 `is_rewatching`

In [40]:
analyze(df_rt['is_rewatching'])


  Nome serie                     is_rewatching
  dtype                          float64
  Dimensione                     123,335,305
  Range indice                   0 … 123335304
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,335,305
  Valori non nulli               119,560,277  (96.94%)
  Null / NaN                     3,775,028  (3.06%)
  Zeri                           119,457,481  (96.86%)
  Positivi                       102,796   (0.08%)
  Negativi                       0   (0.00%)
  Valori unici                   2  (0.00%)
  Valori float                   0

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0.0000
  Max                            1.0000
  Range                          1.0000
  Media                          0.0009
  Mediana                   

**Pulizia necessaria:**  
- Dtype `float64` con valori `0.0`, `1.0` e `NaN` → conversione a booleano nullable (`boolean`)
- I ~3.797.321 null (~3%) sono strutturali: il campo non è compilato per tutti gli entry (assente per `plan_to_watch` e alcune versioni precedenti dell'API MAL)

In [41]:
print('Valori is_rewatching prima della conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

df_rt['is_rewatching'] = df_rt['is_rewatching'].astype('boolean')

print(f'\nis_rewatching dtype : {df_rt["is_rewatching"].dtype}')
print('Valori dopo la conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

Valori is_rewatching prima della conversione:
is_rewatching
0.0    119457481
NaN      3775028
1.0       102796
Name: count, dtype: int64

is_rewatching dtype : boolean
Valori dopo la conversione:
is_rewatching
False    119457481
<NA>       3775028
True        102796
Name: count, dtype: Int64


### 2.6 `num_watched_episodes`

In [42]:
analyze(df_rt['num_watched_episodes'])


  Nome serie                     num_watched_episodes
  dtype                          int64
  Dimensione                     123,335,305
  Range indice                   0 … 123335304
  Tipo                           NUMERICO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   123,335,305
  Valori non nulli               123,335,305  (100.00%)
  Null / NaN                     0  (0.00%)
  Zeri                           35,641,162  (28.90%)
  Positivi                       87,694,143   (71.10%)
  Negativi                       0   (0.00%)
  Valori unici                   1,912  (0.00%)

Statistiche descrittive
────────────────────────────────────────────────────────────────────────────────

  Min                            0
  Max                            65,535
  Range                          65,535
  Media                          12.6593
  Mediana                        4.0000
  Moda/e              

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. Il valore `0` è atteso per entry con `status = 'plan_to_watch'` o per anime non ancora iniziati. Il valore massimo `65535` (= 2¹⁶ − 1) è un artefatto del tipo intero a 16 bit usato dall'API MAL per questo campo, ma riguarda un numero trascurabile di righe.

### 2.7 Chiave composita `(username, anime_id)`

La combinazione `(username, anime_id)` identifica univocamente una riga: ogni utente ha al più un entry per anime. Verifichiamo su un campione di 2 milioni di righe.

In [43]:
sample_size = 2_000_000
df_sample = df_rt.sample(n=sample_size, random_state=42)
n_pk_dup = df_sample.duplicated(subset=['username', 'anime_id'], keep=False).sum()
print(f'Duplicati su (username, anime_id) nel campione di {sample_size:,} righe: {n_pk_dup}')

if n_pk_dup == 0:
    print('→ Chiave composita univoca nel campione, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati, verificare sull\'intero dataset.')

Duplicati su (username, anime_id) nel campione di 2,000,000 righe: 0
→ Chiave composita univoca nel campione, nessuna operazione richiesta.


La chiave composita `(username, anime_id)` è univoca nel campione verificato. Il dataset è pronto per il salvataggio.

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [44]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>15,}')
print(f'Righe dopo cleaning  : {len(df_rt):>15,}')
print(f'Righe rimosse totali : {n_originale - len(df_rt):>15,}')
print()
df_rt.to_csv('../datasets_cleaned/ratings_clean.csv', index=False)
print('Salvato: datasets_cleaned/ratings_clean.csv')

=== Riepilogo Dataset Pulito ===
Righe originali      :     124,298,357
Righe dopo cleaning  :     123,335,305
Righe rimosse totali :         963,052

Salvato: datasets_cleaned/ratings_clean.csv
